In [2]:
import zipfile
import os
from google.colab import files

print("Please select your dataset ZIP file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith(".zip"):
        print(f"Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(".")
        print("Extraction complete!")

for root, dirs, filenames in os.walk("."):
    for file in filenames:
        if file.endswith(".csv") and root != ".":
            source_path = os.path.join(root, file)
            target_path = os.path.join(".", file)
            os.rename(source_path, target_path)

print("\nAvailable CSV files in current working directory:")
print([f for f in os.listdir(".") if f.endswith(".csv")])

Please select your dataset ZIP file:


Saving Assessment 2-20261030T033730Z-1-001.zip to Assessment 2-20261030T033730Z-1-001 (1).zip
Extracting Assessment 2-20261030T033730Z-1-001 (1).zip...
Extraction complete!

Available CSV files in current working directory:
['watch_sessions_80000.csv', 'watch_sessions_extra_10000.csv', 'content_catalog_120.csv', 'subscription_plans_4.csv', 'subscriber_master_2400.csv', 'device_master_8.csv']


In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel

spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()

def read_csv(name):
    return spark.read.csv(name, header=True, inferSchema=True)

sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")

PART A

In [7]:
sessions_df.printSchema()

print("Row Count:", sessions_df.count())

sessions_df.show(10, truncate=False)


root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

Row Count: 80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+---

In [42]:
numeric_fields = [f.name for f in sessions_df.schema.fields if f.dataType.typeName() in ['integer', 'double', 'long', 'float']]
string_fields = [f.name for f in sessions_df.schema.fields if f.dataType.typeName() == 'string']
timestamp_fields = [f.name for f in sessions_df.schema.fields if f.dataType.typeName() in ['timestamp', 'date']]

print("Numeric Fields:", numeric_fields)
print("String Fields:", string_fields)
print("Timestamp Fields:", timestamp_fields)

Numeric Fields: ['watch_minutes', 'completion_pct', 'buffering_seconds', 'data_used_mb', 'user_rating']
String Fields: ['session_id', 'subscriber_id', 'content_id', 'device_id', 'stream_quality', 'session_status']
Timestamp Fields: ['session_start']


In [43]:
q02_df = sessions_df.select("session_id", "subscriber_id", "content_id", "watch_minutes", "session_status") \
                    .filter((F.col("session_status") == "Completed") & (F.col("watch_minutes") > 60))

q02_df.show(5)

+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|     SUB01859|    CT0075|          125|     Completed|
|WS00000179|     SUB00935|    CT0054|           62|     Completed|
|WS00000184|     SUB00072|    CT0119|           91|     Completed|
|WS00000222|     SUB01603|    CT0103|          116|     Completed|
|WS00000233|     SUB01233|    CT0027|          158|     Completed|
+----------+-------------+----------+-------------+--------------+
only showing top 5 rows


In [11]:
q03_df = sessions_df.withColumn("watch_hours", F.col("watch_minutes") / 60.0) \
                    .withColumn("high_engagement",
                                F.when(F.col("completion_pct").isNull(), None)
                                 .when(F.col("completion_pct") >= 75, True)
                                 .otherwise(False))

q03_df.select("session_id", "watch_minutes", "watch_hours", "completion_pct", "high_engagement").show(10)

+----------+-------------+-------------------+--------------+---------------+
|session_id|watch_minutes|        watch_hours|completion_pct|high_engagement|
+----------+-------------+-------------------+--------------+---------------+
|WS00000001|            6|                0.1|          11.8|          false|
|WS00000002|           31| 0.5166666666666667|          56.4|          false|
|WS00000003|           29|0.48333333333333334|          63.0|          false|
|WS00000004|           69|               1.15|          75.0|           true|
|WS00000005|           36|                0.6|          23.4|          false|
|WS00000006|           20| 0.3333333333333333|          19.8|          false|
|WS00000007|           78|                1.3|          83.9|           true|
|WS00000008|           22|0.36666666666666664|          66.7|          false|
|WS00000009|           21|               0.35|          80.8|           true|
|WS00000010|           21|               0.35|          65.6|   

In [12]:
q03_df = sessions_df.withColumn("watch_hours", F.col("watch_minutes") / 60.0) \
                    .withColumn("high_engagement",
                                F.when(F.col("completion_pct").isNull(), None)
                                 .when(F.col("completion_pct") >= 75, True)
                                 .otherwise(False))

q03_df.select("session_id", "watch_minutes", "watch_hours", "completion_pct", "high_engagement").show(10)

+----------+-------------+-------------------+--------------+---------------+
|session_id|watch_minutes|        watch_hours|completion_pct|high_engagement|
+----------+-------------+-------------------+--------------+---------------+
|WS00000001|            6|                0.1|          11.8|          false|
|WS00000002|           31| 0.5166666666666667|          56.4|          false|
|WS00000003|           29|0.48333333333333334|          63.0|          false|
|WS00000004|           69|               1.15|          75.0|           true|
|WS00000005|           36|                0.6|          23.4|          false|
|WS00000006|           20| 0.3333333333333333|          19.8|          false|
|WS00000007|           78|                1.3|          83.9|           true|
|WS00000008|           22|0.36666666666666664|          66.7|          false|
|WS00000009|           21|               0.35|          80.8|           true|
|WS00000010|           21|               0.35|          65.6|   

In [13]:
q03_df = sessions_df.withColumn("watch_hours", F.col("watch_minutes") / 60.0) \
                    .withColumn("high_engagement",
                                F.when(F.col("completion_pct").isNull(), None)
                                 .when(F.col("completion_pct") >= 75, True)
                                 .otherwise(False))

q03_df.select("session_id", "watch_minutes", "watch_hours", "completion_pct", "high_engagement").show(10)

+----------+-------------+-------------------+--------------+---------------+
|session_id|watch_minutes|        watch_hours|completion_pct|high_engagement|
+----------+-------------+-------------------+--------------+---------------+
|WS00000001|            6|                0.1|          11.8|          false|
|WS00000002|           31| 0.5166666666666667|          56.4|          false|
|WS00000003|           29|0.48333333333333334|          63.0|          false|
|WS00000004|           69|               1.15|          75.0|           true|
|WS00000005|           36|                0.6|          23.4|          false|
|WS00000006|           20| 0.3333333333333333|          19.8|          false|
|WS00000007|           78|                1.3|          83.9|           true|
|WS00000008|           22|0.36666666666666664|          66.7|          false|
|WS00000009|           21|               0.35|          80.8|           true|
|WS00000010|           21|               0.35|          65.6|   

In [14]:
q04_df = sessions_df.orderBy(F.col("data_used_mb").desc())
q04_df.show(20)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|     SUB00155|    CT0069|     DV08|2026-07-20 00:11:00|          167|          97.7|              9.0|            4K|     Completed|      8881.8|        3.0|
|WS00074513|     SUB00684|    CT0056|     DV05|2026-08-09 21:53:00|          157|          96.9|              8.0|            4K|     Completed|      8408.4|       NULL|
|WS00039201|     SUB01664|    CT0027|     DV08|2026-05-16 06:35:00|          161|          97.0|              9.0|            4K|     Completed|      

In [ ]:
PART B

In [15]:
target_cols = ["subscriber_id", "device_id", "completion_pct", "buffering_seconds", "user_rating"]

sessions_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in target_cols
]).show()

+-------------+---------+--------------+-----------------+-----------+
|subscriber_id|device_id|completion_pct|buffering_seconds|user_rating|
+-------------+---------+--------------+-----------------+-----------+
|          951|     2003|          3513|             4424|      77605|
+-------------+---------+--------------+-----------------+-----------+



In [16]:
sessions_zero_buff_df = sessions_df.fillna({"buffering_seconds": 0})

In [17]:
sessions_no_null_device_df = sessions_df.filter(F.col("device_id").isNotNull())

print("Original count:", sessions_df.count())
print("Filtered count (no null device_id):", sessions_no_null_device_df.count())

Original count: 80000
Filtered count (no null device_id): 77997


In [18]:
q08_df = sessions_df.withColumn(
    "completion_category",
    F.when(F.col("completion_pct").isNull(), "Unknown")
     .when(F.col("completion_pct") < 30, "Low")
     .when((F.col("completion_pct") >= 30) & (F.col("completion_pct") < 90), "Medium")
     .when(F.col("completion_pct") >= 90, "High")
     .otherwise("Unknown")
)

q08_df.select("session_id", "completion_pct", "completion_category").show(10)

+----------+--------------+-------------------+
|session_id|completion_pct|completion_category|
+----------+--------------+-------------------+
|WS00000001|          11.8|                Low|
|WS00000002|          56.4|             Medium|
|WS00000003|          63.0|             Medium|
|WS00000004|          75.0|             Medium|
|WS00000005|          23.4|                Low|
|WS00000006|          19.8|                Low|
|WS00000007|          83.9|             Medium|
|WS00000008|          66.7|             Medium|
|WS00000009|          80.8|             Medium|
|WS00000010|          65.6|             Medium|
+----------+--------------+-------------------+
only showing top 10 rows


In [ ]:
PART C

In [19]:
q09_df = sessions_df.join(content_df, "content_id", "inner") \
                    .groupBy("genre") \
                    .agg(
                        F.count("session_id").alias("session_count"),
                        F.sum("watch_minutes").alias("total_watch_minutes")
                    )

q09_df.show()

+-----------+-------------+-------------------+
|      genre|session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|   Thriller|         5972|             248430|
|  Adventure|        15561|             839700|
|     Sports|         9967|             383044|
|      Drama|        10733|             519872|
|Documentary|         7432|             256626|
|  Animation|        11886|             490335|
|     Comedy|         9105|             452563|
|     Sci-Fi|         9344|             433946|
+-----------+-------------+-------------------+



In [20]:
q10_df = sessions_df.join(devices_df, "device_id", "inner") \
                    .groupBy("device_type") \
                    .agg(
                        F.avg("buffering_seconds").alias("avg_buffering_seconds"),
                        F.avg("watch_minutes").alias("avg_watch_minutes")
                    )

q10_df.show()

+---------------+---------------------+------------------+
|    device_type|avg_buffering_seconds| avg_watch_minutes|
+---------------+---------------------+------------------+
|         iPhone|   35.761106907361366|45.226752503576535|
|Streaming Stick|   36.320860309448044| 45.08570541351827|
|Desktop Browser|    36.02889125799574|45.571818733562615|
|  Android Phone|   35.676486693680864|  45.0885616509385|
| Laptop Browser|   36.930938301907844| 45.84465713083719|
|         Tablet|    35.23614616481949| 45.38943894389439|
|       Smart TV|    36.67392739273927| 45.33776623376623|
|   Game Console|    35.51243726816496| 45.04469158686026|
+---------------+---------------------+------------------+



In [21]:
q11_df = sessions_df.filter(F.col("subscriber_id").isNotNull()) \
                    .join(subscribers_df, "subscriber_id", "inner") \
                    .join(plans_df, "plan_id", "inner") \
                    .groupBy("plan_name") \
                    .agg(
                        (F.sum(F.when(F.col("session_status") == "completed", 1).otherwise(0)) / F.count("session_id")).alias("completion_rate")
                    )

q11_df.show()

+---------+---------------+
|plan_name|completion_rate|
+---------+---------------+
|  Premium|            0.0|
|    Basic|            0.0|
|   Mobile|            0.0|
| Standard|            0.0|
+---------+---------------+



In [23]:
q12_df = subscribers_df.filter(F.col("subscriber_status") == "Active") \
                       .join(plans_df, "plan_id", "inner") \
                       .groupBy("plan_name") \
                       .agg( \
                           F.countDistinct("subscriber_id").alias("active_subscriber_count"), \
                           F.sum("monthly_fee_inr").alias("estimated_mrr") \
                       )

q12_df.show()

+---------+-----------------------+-------------+
|plan_name|active_subscriber_count|estimated_mrr|
+---------+-----------------------+-------------+
|  Premium|                    326|       211574|
|    Basic|                    632|       157368|
|   Mobile|                    443|        66007|
| Standard|                    604|       240996|
+---------+-----------------------+-------------+



In [24]:
unmatched_subs_count = sessions_df.filter(F.col("subscriber_id").isNotNull()) \
                                  .join(subscribers_df, "subscriber_id", "left_anti") \
                                  .count()
unmatched_devs_count = sessions_df.filter(F.col("device_id").isNotNull()) \
                                  .join(devices_df, "device_id", "left_anti") \
                                  .count()

print("Unmatched subscriber records count:", unmatched_subs_count)
print("Unmatched device records count:", unmatched_devs_count)

Unmatched subscriber records count: 571
Unmatched device records count: 481


PART D

In [25]:
sessions_ts_df = sessions_df.withColumn("session_start", F.to_timestamp("session_start"))

q14_df = sessions_ts_df.filter(F.year("session_start") == 2026) \
                       .groupBy(F.month("session_start").alias("month")) \
                       .agg(
                           F.count("session_id").alias("monthly_session_count"),
                           F.sum("watch_minutes").alias("total_watch_minutes")
                       ) \
                       .orderBy("month")

q14_df.show()

+-----+---------------------+-------------------+
|month|monthly_session_count|total_watch_minutes|
+-----+---------------------+-------------------+
|    1|                 9070|             407215|
|    2|                 8218|             372726|
|    3|                 9019|             403971|
|    4|                 8817|             399432|
|    5|                 9006|             407386|
|    6|                 8776|             404203|
|    7|                 9204|             416104|
|    8|                 8996|             407374|
|    9|                 8894|             406105|
+-----+---------------------+-------------------+



In [26]:
q15_df = sessions_ts_df.filter(F.col("subscriber_id").isNotNull()) \
                       .join(subscribers_df, "subscriber_id", "inner") \
                       .withColumn("signup_date", F.to_date("signup_date")) \
                       .withColumn("days_since_signup", F.datediff(F.to_date("session_start"), F.col("signup_date"))) \
                       .orderBy("session_start")

q15_df.select("session_id", "subscriber_id", "signup_date", "session_start", "days_since_signup").show(10)

+----------+-------------+-----------+-------------------+-----------------+
|session_id|subscriber_id|signup_date|      session_start|days_since_signup|
+----------+-------------+-----------+-------------------+-----------------+
|WS00035235|     SUB01707| 2024-02-13|2026-01-01 00:00:00|              688|
|WS00074739|     SUB02206| 2025-08-31|2026-01-01 00:04:00|              123|
|WS00032058|     SUB01248| 2024-03-17|2026-01-01 00:04:00|              655|
|WS00048560|     SUB02342| 2025-08-25|2026-01-01 00:14:00|              129|
|WS00067087|     SUB00563| 2024-08-11|2026-01-01 00:15:00|              508|
|WS00004245|     SUB00711| 2024-10-26|2026-01-01 00:18:00|              432|
|WS00041710|     SUB00261| 2025-05-15|2026-01-01 00:21:00|              231|
|WS00045132|     SUB02167| 2025-11-22|2026-01-01 00:28:00|               40|
|WS00036438|     SUB01237| 2024-02-01|2026-01-01 00:41:00|              700|
|WS00030050|     SUB00646| 2024-11-20|2026-01-01 00:51:00|              407|

In [27]:
q16_df = sessions_ts_df.withColumn("day_type",
                                   F.when(F.dayofweek("session_start").isin(1, 7), "Weekend")
                                    .otherwise("Weekday")) \
                       .groupBy("day_type") \
                       .agg(F.avg("watch_minutes").alias("avg_watch_minutes"))

q16_df.show()

+--------+------------------+
|day_type| avg_watch_minutes|
+--------+------------------+
| Weekday| 45.26460823373174|
| Weekend|45.411601967328295|
+--------+------------------+



In [29]:
genre_title_agg = sessions_df.join(content_df, "content_id", "inner") \
                             .groupBy("genre", "content_title") \
                             .agg(F.sum("watch_minutes").alias("total_watch_minutes"))

window_spec_genre = Window.partitionBy("genre").orderBy(F.col("total_watch_minutes").desc())

q17_df = genre_title_agg.withColumn("rank", F.dense_rank().over(window_spec_genre)) \
                        .filter(F.col("rank") <= 3)

q17_df.show(20)

+-----------+----------------+-------------------+----+
|      genre|   content_title|total_watch_minutes|rank|
+-----------+----------------+-------------------+----+
|  Adventure|Infinite Horizon|              66395|   1|
|  Adventure|    Lost Journey|              61823|   2|
|  Adventure|    Solar Memory|              60540|   3|
|  Animation|      Last River|              51398|   1|
|  Animation| Emerald Circuit|              51310|   2|
|  Animation|   Golden Puzzle|              50515|   3|
|     Comedy|    Blue Horizon|              61428|   1|
|     Comedy|   Bright Valley|              60921|   2|
|     Comedy|   Silver Valley|              60422|   3|
|Documentary|   Fading Signal|              47085|   1|
|Documentary|  Emerald Valley|              40819|   2|
|Documentary|   Solar Horizon|              36152|   3|
|      Drama|  Crimson Summit|              65724|   1|
|      Drama|  Rising Archive|              57301|   2|
|      Drama|     Wild Harbor|              4547

In [30]:
window_spec_sub = Window.partitionBy("subscriber_id").orderBy(F.col("session_start").desc(), F.col("session_id").desc())

q18_df = sessions_ts_df.filter(F.col("subscriber_id").isNotNull()) \
                       .withColumn("rn", F.row_number().over(window_spec_sub)) \
                       .filter(F.col("rn") == 1) \
                       .drop("rn")

q18_df.show(10)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00020822|     SUB00001|    CT0073|     DV03|2026-09-11 19:19:00|           34|          89.5|            145.0|            HD|   Interrupted|       418.3|       NULL|
|WS00032744|     SUB00002|    CT0076|     DV05|2026-09-02 05:41:00|           45|          73.8|            219.0|       Full HD|   Interrupted|      1184.1|       NULL|
|WS00006110|     SUB00003|    CT0004|     DV03|2026-09-30 03:13:00|           11|          15.3|              3.0|            4K|     Abandoned|      

In [31]:
window_spec_lag = Window.partitionBy("subscriber_id").orderBy("session_start", "session_id")

q19_df = sessions_ts_df.filter(F.col("subscriber_id").isNotNull()) \
                       .withColumn("prev_watch_minutes", F.lag("watch_minutes").over(window_spec_lag)) \
                       .withColumn("watch_minutes_diff", F.col("watch_minutes") - F.col("prev_watch_minutes"))

q19_df.select("subscriber_id", "session_id", "session_start", "watch_minutes", "prev_watch_minutes", "watch_minutes_diff").show(15)

+-------------+----------+-------------------+-------------+------------------+------------------+
|subscriber_id|session_id|      session_start|watch_minutes|prev_watch_minutes|watch_minutes_diff|
+-------------+----------+-------------------+-------------+------------------+------------------+
|     SUB00001|WS00064390|2026-01-02 17:32:00|           58|              NULL|              NULL|
|     SUB00001|WS00045183|2026-01-05 04:51:00|           78|                58|                20|
|     SUB00001|WS00028484|2026-01-09 08:46:00|            4|                78|               -74|
|     SUB00001|WS00043555|2026-01-24 19:41:00|          108|                 4|               104|
|     SUB00001|WS00041862|2026-02-18 07:43:00|           79|               108|               -29|
|     SUB00001|WS00059616|2026-02-25 07:38:00|           18|                79|               -61|
|     SUB00001|WS00042345|2026-03-13 07:34:00|           44|                18|                26|
|     SUB0

PART E

In [32]:
union_sessions_df = sessions_df.unionByName(extra_df)
print("Total rows after union:", union_sessions_df.count())

dedup_sessions_df = union_sessions_df.dropDuplicates(["session_id"])
print("Unique rows after deduplication:", dedup_sessions_df.count())

Total rows after union: 90000
Unique rows after deduplication: 89400


In [36]:
lazy_pipeline = dedup_sessions_df.filter(F.col("session_status") == "completed") \
                                 .join(content_df, "content_id", "inner") \
                                 .withColumn("watch_hours", F.col("watch_minutes") / 60.0) \
                                 .groupBy("genre") \
                                 .agg(F.sum("watch_hours").alias("total_hours")) \
                                 .orderBy(F.col("total_hours").desc())

lazy_pipeline.explain(True)
lazy_pipeline.show()

== Parsed Logical Plan ==
'Sort ['total_hours DESC NULLS LAST], true
+- Aggregate [genre#100], [genre#100, sum(watch_hours#3468) AS total_hours#3469]
   +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, content_type#101, duration_minutes#102, release_year#103, content_language#104, (cast(watch_minutes#22 as double) / cast(60.0 as double)) AS watch_hours#3468]
      +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, content_type#101, duration_minutes#102, release_year#103, content_language#104]
         +- Join Inner, (content_id#19 = content_id#98)
            :- Filter (session_status#26 = c

In [39]:
# Use F.lower() to ensure case-insensitive matching for 'completed' status
lazy_pipeline = dedup_sessions_df.filter(F.lower(F.col("session_status")) == "completed") \
                                 .join(content_df, "content_id", "inner") \
                                 .withColumn("watch_hours", F.col("watch_minutes") / 60.0) \
                                 .groupBy("genre") \
                                 .agg(F.sum("watch_hours").alias("total_hours")) \
                                 .orderBy(F.col("total_hours").desc())

# Inspect the query execution plan without triggering an action
lazy_pipeline.explain(True)

# Trigger action to display total hours per genre
lazy_pipeline.show()

== Parsed Logical Plan ==
'Sort ['total_hours DESC NULLS LAST], true
+- Aggregate [genre#100], [genre#100, sum(watch_hours#3963) AS total_hours#3964]
   +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, content_type#101, duration_minutes#102, release_year#103, content_language#104, (cast(watch_minutes#22 as double) / cast(60.0 as double)) AS watch_hours#3963]
      +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, content_type#101, duration_minutes#102, release_year#103, content_language#104]
         +- Join Inner, (content_id#19 = content_id#98)
            :- Filter (lower(session_status#

In [40]:
import time

# Cache completed sessions using case-insensitive status matching
completed_sessions_df = dedup_sessions_df.filter(F.lower(F.col("session_status")) == "completed").cache()

# First execution (computes and caches data into memory)
t0 = time.time()
c1 = completed_sessions_df.count()
a1 = completed_sessions_df.groupBy("stream_quality").agg(F.avg("watch_minutes")).collect()
t1 = time.time()
print(f"First run execution time: {t1 - t0:.4f} seconds")

# Second execution (reads directly from in-memory cache)
t2 = time.time()
c2 = completed_sessions_df.count()
a2 = completed_sessions_df.groupBy("stream_quality").agg(F.avg("watch_minutes")).collect()
t3 = time.time()
print(f"Second run execution time: {t3 - t2:.4f} seconds")

# Unpersist DataFrame to free memory
completed_sessions_df.unpersist()

First run execution time: 13.6554 seconds
Second run execution time: 5.0126 seconds


DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [41]:
# Filter for 4K sessions using case-insensitive matching (matches '4K', '4k', etc.)
sessions_4k_df = dedup_sessions_df.filter(F.lower(F.col("stream_quality")) == "4k")

# Persist with MEMORY_AND_DISK storage level
sessions_4k_df.persist(StorageLevel.MEMORY_AND_DISK)

# Action 1: Count of 4K sessions
print("4K Session Count:", sessions_4k_df.count())

# Action 2: Average data used per session status for 4K sessions
sessions_4k_df.groupBy("session_status").agg(F.avg("data_used_mb")).show()

# Free memory and disk space
sessions_4k_df.unpersist()

4K Session Count: 8127
+--------------+------------------+
|session_status| avg(data_used_mb)|
+--------------+------------------+
|     Completed|3718.9285932721723|
|     Abandoned| 702.8057020669992|
|   Interrupted|2364.4421416803934|
+--------------+------------------+



DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [37]:
repart_df = dedup_sessions_df.repartition(8)

repart_df.withColumn("partition_id", F.spark_partition_id()) \
         .groupBy("partition_id") \
         .count() \
         .orderBy("partition_id") \
         .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|11175|
|           1|11175|
|           2|11175|
|           3|11175|
|           4|11175|
|           5|11175|
|           6|11175|
|           7|11175|
+------------+-----+



In [38]:
print("--- Coalesce(2) Plan ---")
repart_df.coalesce(2).explain()

print("--- Repartition(2) Plan ---")
repart_df.repartition(2).explain()

# Write coalesce(2) output to Parquet
repart_df.coalesce(2).write.mode("overwrite").parquet("output_sessions_parquet")

--- Coalesce(2) Plan ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=3534]
      +- SortAggregate(key=[session_id#17], functions=[first(subscriber_id#18, false), first(content_id#19, false), first(device_id#20, false), first(session_start#21, false), first(watch_minutes#22, false), first(completion_pct#23, false), first(buffering_seconds#24, false), first(stream_quality#25, false), first(session_status#26, false), first(data_used_mb#27, false), first(user_rating#28, false)])
         +- Sort [session_id#17 ASC NULLS FIRST], false, 0
            +- Exchange hashpartitioning(session_id#17, 200), ENSURE_REQUIREMENTS, [plan_id=3531]
               +- SortAggregate(key=[session_id#17], functions=[partial_first(subscriber_id#18, false), partial_first(content_id#19, false), partial_first(device_id#20, false), partial_first(session_start#21, false), partial_first(watch_minutes#22, false), partial_f

Final Capstone - Streaming Performance Report

In [47]:
all_sessions_df = sessions_df.unionByName(extra_df).dropDuplicates(["session_id"])

In [48]:
cleaned_sessions_df = all_sessions_df.fillna({"buffering_seconds": 0, "data_used_mb": 0.0}) \
                                     .filter(F.col("device_id").isNotNull()) \
                                     .withColumn("session_start", F.to_timestamp("session_start"))

In [49]:
full_df = cleaned_sessions_df.filter(F.col("subscriber_id").isNotNull()) \
                             .join(subscribers_df, "subscriber_id", "inner") \
                             .join(content_df, "content_id", "inner") \
                             .join(devices_df, "device_id", "inner") \
                             .join(plans_df, "plan_id", "inner")

In [50]:
monthly_metrics_df = full_df.withColumn("year_month", F.date_format("session_start", "yyyy-MM")) \
                            .groupBy("year_month", "region", "genre") \
                            .agg(
                                F.count("session_id").alias("session_count"),
                                F.sum(F.col("watch_minutes") / 60.0).alias("watch_hours"),
                                (F.sum(F.when(F.lower(F.col("session_status")) == "completed", 1).otherwise(0)) / F.count("session_id")).alias("completion_rate"),
                                F.avg("buffering_seconds").alias("avg_buffering_seconds"),
                                F.sum("data_used_mb").alias("total_data_used_mb")
                            )

In [51]:
title_col = "content_title" if "content_title" in full_df.columns else "title"

title_watch_df = full_df.groupBy("genre", title_col) \
                        .agg(F.sum(F.col("watch_minutes") / 60.0).alias("total_watch_hours"))
genre_window = Window.partitionBy("genre").orderBy(F.col("total_watch_hours").desc())
top_3_content_df = title_watch_df.withColumn("rank", F.dense_rank().over(genre_window)) \
                                 .filter(F.col("rank") <= 3)

In [52]:
monthly_metrics_df.cache()
print("Monthly Metrics Record Count:", monthly_metrics_df.count())
monthly_metrics_df.show(10)
top_3_content_df.show(20)
monthly_metrics_df.unpersist()

Monthly Metrics Record Count: 360
+----------+-------+-----------+-------------+------------------+-------------------+---------------------+------------------+
|year_month| region|      genre|session_count|       watch_hours|    completion_rate|avg_buffering_seconds|total_data_used_mb|
+----------+-------+-----------+-------------+------------------+-------------------+---------------------+------------------+
|   2026-03|   West|     Sci-Fi|          300|237.78333333333322|0.10333333333333333|   33.516666666666666|          241809.7|
|   2026-06|  North|  Animation|          309|212.24999999999991|0.10032362459546926|    34.16504854368932|239601.10000000012|
|   2026-07|  North|Documentary|          201| 122.4666666666667|0.07960199004975124|   31.955223880597014|          138246.7|
|   2026-07|  North|     Comedy|          246|193.43333333333334|0.07723577235772358|     31.5609756097561|          215851.8|
|   2026-09|   East|  Adventure|          263|233.06666666666672|0.0798479087

DataFrame[year_month: string, region: string, genre: string, session_count: bigint, watch_hours: double, completion_rate: double, avg_buffering_seconds: double, total_data_used_mb: double]

In [53]:
monthly_metrics_df.coalesce(3).write.mode("overwrite").parquet("final_streaming_performance_report_parquet")

In [54]:
report_df = spark.read.parquet("final_streaming_performance_report_parquet")
print("Total Records in Saved Parquet Report:", report_df.count())
report_df.show(20, truncate=False)

Total Records in Saved Parquet Report: 360
+----------+-------+-----------+-------------+------------------+-------------------+---------------------+------------------+
|year_month|region |genre      |session_count|watch_hours       |completion_rate    |avg_buffering_seconds|total_data_used_mb|
+----------+-------+-----------+-------------+------------------+-------------------+---------------------+------------------+
|2026-03   |West   |Sci-Fi     |300          |237.78333333333322|0.10333333333333333|33.516666666666666   |241809.7          |
|2026-06   |North  |Animation  |309          |212.24999999999991|0.10032362459546926|34.16504854368932    |239601.10000000012|
|2026-07   |North  |Documentary|201          |122.4666666666667 |0.07960199004975124|31.955223880597014   |138246.7          |
|2026-07   |North  |Comedy     |246          |193.43333333333334|0.07723577235772358|31.5609756097561     |215851.8          |
|2026-09   |East   |Adventure  |263          |233.06666666666672|0.0